# 01 - Data Collection: Kaggle & OIV Datasets

**Objective**: Download and perform initial inspection of primary data sources for Spanish Wine & Spirits market analysis.

**Data Sources**:
1. Kaggle - Spanish Wine Quality Dataset (product-level)
2. OIV - Time series data (production, consumption, exports)
3. USDA - Industry reports (manual download)

**Author**: [Your Name]  
**Date**: April 2026  
**Project**: Spanish Wine & Spirits Market Analysis

## 1. Setup & Imports

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data collection
import requests
from kaggle.api.kaggle_api_extended import KaggleApi

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Create Project Directory Structure

In [ ]:
# Define directory structure
directories = [
    '../data/raw',
    '../data/processed',
    '../data/external',
    '../reports/figures',
    '../models',
    '../src'
]

# Create directories
for directory in directories:
    Path(directory).mkdir(parents=True, exist_ok=True)
    
print("✅ Directory structure created successfully")

# Create .gitkeep files to preserve empty directories
for directory in directories:
    gitkeep_path = Path(directory) / '.gitkeep'
    gitkeep_path.touch(exist_ok=True)

## 3. Download Kaggle Dataset

**Dataset**: Spanish Wine Quality Dataset  
**URL**: https://www.kaggle.com/datasets/fedesoriano/spanish-wine-quality-dataset

### Prerequisites:
1. Create Kaggle account
2. Go to Account Settings → API → Create New API Token
3. This downloads `kaggle.json` file
4. Place it in `~/.kaggle/` directory
5. Set permissions: `chmod 600 ~/.kaggle/kaggle.json`

In [ ]:
# Initialize Kaggle API
try:
    api = KaggleApi()
    api.authenticate()
    print("✅ Kaggle API authenticated successfully")
    
    # Download dataset
    dataset_name = 'fedesoriano/spanish-wine-quality-dataset'
    download_path = '../data/raw'
    
    print(f"📥 Downloading dataset: {dataset_name}...")
    api.dataset_download_files(dataset_name, path=download_path, unzip=True)
    print("✅ Dataset downloaded successfully")
    
except Exception as e:
    print(f"❌ Error downloading from Kaggle: {e}")
    print("\n📝 Manual download instructions:")
    print("1. Visit: https://www.kaggle.com/datasets/fedesoriano/spanish-wine-quality-dataset")
    print("2. Download the dataset")
    print("3. Extract to '../data/raw/' directory")

## 4. Load and Inspect Kaggle Dataset

In [ ]:
# Load the dataset
try:
    wine_df = pd.read_csv('../data/raw/wines_SPA.csv')
    print("✅ Dataset loaded successfully")
    print(f"\nDataset shape: {wine_df.shape}")
    print(f"Number of wines: {wine_df.shape[0]:,}")
    print(f"Number of features: {wine_df.shape[1]}")
except FileNotFoundError:
    print("❌ File not found. Please download the dataset manually.")
    wine_df = None

In [ ]:
if wine_df is not None:
    print("\n📊 COLUMN OVERVIEW")
    print("=" * 80)
    print(wine_df.columns.tolist())
    
    print("\n📋 DATA INFO")
    print("=" * 80)
    wine_df.info()

In [ ]:
if wine_df is not None:
    print("\n🔍 FIRST 5 ROWS")
    print("=" * 80)
    display(wine_df.head())

In [ ]:
if wine_df is not None:
    print("\n📈 DESCRIPTIVE STATISTICS")
    print("=" * 80)
    display(wine_df.describe())

In [ ]:
if wine_df is not None:
    print("\n❓ MISSING VALUES ANALYSIS")
    print("=" * 80)
    
    missing = wine_df.isnull().sum()
    missing_pct = (missing / len(wine_df)) * 100
    missing_df = pd.DataFrame({
        'Column': missing.index,
        'Missing_Count': missing.values,
        'Missing_Percentage': missing_pct.values
    })
    missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
    
    if len(missing_df) > 0:
        display(missing_df)
    else:
        print("✅ No missing values detected")

## 5. OIV Time Series Data Collection

**Source**: International Organisation of Vine and Wine (OIV)  
**URL**: https://www.oiv.int/what-we-do/statistics

### Manual Download Required:
1. Visit the OIV statistics page
2. Download the following datasets:
   - Wine production by country (2011-2023)
   - Wine consumption by country
   - International wine trade statistics
3. Save to `../data/external/` directory

**Note**: OIV data may require manual extraction from PDF reports or Excel files.

In [ ]:
# Placeholder for OIV data
print("📝 OIV Data Collection Notes:")
print("=" * 80)
print("1. Visit: https://www.oiv.int/what-we-do/statistics")
print("2. Download relevant reports (production, consumption, trade)")
print("3. Extract data tables to Excel/CSV format")
print("4. Save to ../data/external/ directory")
print("\n⚠️  Manual processing required - OIV provides data in various formats")
print("\nRecommended files:")
print("  - State of the World Vitivinicultural Sector (annual report)")
print("  - Statistical Report on World Vitiviniculture")
print("  - Focus on wine production, consumption, and trade for Spain")

### Sample OIV Data Structure

Create template for manually extracted OIV data:

In [ ]:
# Template for OIV time series data
oiv_template = {
    'Year': [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023],
    'Production_Spain_Mhl': [np.nan] * 13,  # Million hectoliters
    'Consumption_Spain_Mhl': [np.nan] * 13,
    'Exports_Spain_Mhl': [np.nan] * 13,
    'Imports_Spain_Mhl': [np.nan] * 13,
    'Vineyard_Area_1000ha': [np.nan] * 13,  # Thousand hectares
}

oiv_df_template = pd.DataFrame(oiv_template)

# Save template
oiv_df_template.to_csv('../data/external/oiv_spain_template.csv', index=False)
print("✅ OIV data template created: ../data/external/oiv_spain_template.csv")
print("\n📝 Fill in this template with data from OIV reports")
display(oiv_df_template)

## 6. USDA Reports Collection

**Source**: USDA Foreign Agricultural Service  
**Reports**: Spain Wine Sector Reports (2020, 2024)

### Download Links:
- [Spain Wine Sector Outlook 2024](https://apps.fas.usda.gov/newgainapi/api/Report/DownloadReportByFileName?fileName=Spain+Wine+Sector+Outlook+2024_Madrid_Spain_SP2024-0023.pdf)
- [Spanish Wine Sector Update 2020](https://apps.fas.usda.gov/newgainapi/api/Report/DownloadReportByFileName?fileName=Spanish+Wine+Sector+Update_Madrid_Spain_06-28-2020)

In [ ]:
# Download USDA reports
usda_reports = {
    'Spain_Wine_Sector_2024.pdf': 'https://apps.fas.usda.gov/newgainapi/api/Report/DownloadReportByFileName?fileName=Spain+Wine+Sector+Outlook+2024_Madrid_Spain_SP2024-0023.pdf',
    'Spain_Wine_Update_2020.pdf': 'https://apps.fas.usda.gov/newgainapi/api/Report/DownloadReportByFileName?fileName=Spanish+Wine+Sector+Update_Madrid_Spain_06-28-2020'
}

for filename, url in usda_reports.items():
    filepath = f'../data/external/{filename}'
    
    try:
        print(f"📥 Downloading: {filename}...")
        response = requests.get(url, timeout=30)
        
        if response.status_code == 200:
            with open(filepath, 'wb') as f:
                f.write(response.content)
            print(f"✅ Downloaded successfully: {filename}")
        else:
            print(f"❌ Failed to download {filename} (Status: {response.status_code})")
            
    except Exception as e:
        print(f"❌ Error downloading {filename}: {e}")
        print(f"   Manual download: {url}")

## 7. Data Collection Summary

In [ ]:
print("\n" + "=" * 80)
print("📊 DATA COLLECTION SUMMARY")
print("=" * 80)

# Check what files exist
raw_files = list(Path('../data/raw').glob('*'))
external_files = list(Path('../data/external').glob('*'))

print("\n✅ KAGGLE DATASET (Product-level)")
if wine_df is not None:
    print(f"   Status: LOADED")
    print(f"   Records: {len(wine_df):,}")
    print(f"   Features: {wine_df.shape[1]}")
    print(f"   File: wines_SPA.csv")
else:
    print(f"   Status: NOT LOADED (manual download required)")

print("\n📁 RAW DATA FILES:")
for f in raw_files:
    if f.name != '.gitkeep':
        print(f"   - {f.name}")

print("\n📁 EXTERNAL DATA FILES:")
for f in external_files:
    if f.name != '.gitkeep':
        print(f"   - {f.name}")

print("\n📋 NEXT STEPS:")
print("   1. ✅ Complete data collection (fill OIV template)")
   2. Run notebook 02: Exploratory Data Analysis (EDA)")
print("   3. Clean and process data for modeling")
print("   4. Begin time series analysis")

## 8. Notes & Observations

### Data Quality Considerations:
- [ ] Check for duplicate entries in Kaggle dataset
- [ ] Validate price ranges (outliers?)
- [ ] Standardize region names
- [ ] Handle missing values strategy

### OIV Data Requirements:
- [ ] Spanish production (2011-2023)
- [ ] Spanish consumption (2011-2023)
- [ ] Export volumes by destination
- [ ] Global context (France, Italy comparison)

### LVMH-Specific Data Needed:
- [ ] LVMH annual reports (Wine & Spirits segment)
- [ ] Numanthia production/pricing data
- [ ] Chandon España market share
- [ ] Premium segment benchmarks